### Set up

In [1]:
import pandas as pd
import requests
import json
import dotenv
import os, time, requests
dotenv.load_dotenv()

True

In [2]:
key = os.getenv('key')
token = os.getenv("token")

In [3]:
#user agent

botname = 'ds6021'
version = '0.0'
website = 'Class project'
email = 'qny2gg@virginia.edu'

useragent = f'{botname}/{version} ({website}; {email}) python-requests/{requests.__version__}'
useragent

'ds6021/0.0 (Class project; qny2gg@virginia.edu) python-requests/2.34.2'

In [4]:
BASE = "https://api.themoviedb.org/3"

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {token}",
    "accept": "application/json",
    "User-Agent": useragent,
})

def get(path, **params):
    for attempt in range(5):
        r = session.get(f"{BASE}{path}", params=params, timeout=20)
        if r.status_code == 429:               
            time.sleep(int(r.headers.get("Retry-After", 2)))
            continue
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()
    return None

### Collect movie IDs by year

Loops through each release year from 1990 to 2025 collecting the IDs of movies that have at least 100 votes, most-voted first. It stores those IDs in a deduplicated list

Selection bias: films were picked by current vote count, which is post-release information, so flops are under-represented. 

In [5]:
def collect_ids(years, min_votes=100, pages_per_year=25):
    ids = {}
    for yr in years:
        for page in range(1, pages_per_year + 1):
            data = get("/discover/movie",
                       primary_release_year=yr,
                       **{"vote_count.gte": min_votes},
                       sort_by="vote_count.desc",
                       language="en-US",
                       page=page)
            if not data or not data["results"]:
                break
            for m in data["results"]:
                ids[m["id"]] = yr
            if page >= data["total_pages"]:
                break
            time.sleep(0.05)
        print(yr, len(ids))
    return list(ids)

movie_ids = collect_ids(range(1990, 2026))
len(movie_ids)

1990 201
1991 405
1992 606
1993 828
1994 1067
1995 1305
1996 1549
1997 1795
1998 2074
1999 2373
2000 2664
2001 2978
2002 3324
2003 3698
2004 4107
2005 4525
2006 5025
2007 5525
2008 6025
2009 6525
2010 7025
2011 7525
2012 8025
2013 8525
2014 9025
2015 9525
2016 10025
2017 10525
2018 11025
2019 11525
2020 12025
2021 12525
2022 13025
2023 13525
2024 14025
2025 14509


14509

### Pull details of the movie

Takes each movie ID from above andpulls that movie’s full details, then saves everything as a table

In [6]:
def us_certification(d):
    # Returns the US rating (G, PG, PG-13, R, NC-17) from the release_dates data
    countries = d.get("release_dates", {}).get("results", [])
    for country in countries:
        if country["iso_3166_1"] != "US":
            continue
        # Prefer the theatrical release (type 3), otherwise take any non-blank rating
        fallback = None
        for release in country["release_dates"]:
            cert = release.get("certification", "").strip()
            if cert == "":
                continue
            if release.get("type") == 3:
                return cert
            if fallback is None:
                fallback = cert
        return fallback
    return None

def parse_movie(d):
    crew = d.get("credits", {}).get("crew", [])
    cast = d.get("credits", {}).get("cast", [])
    directors = [c["name"] for c in crew if c["job"] == "Director"]
    collection = d.get("belongs_to_collection") or {}   # None when the film is not in a collection
    return {
        "tmdb_id": d["id"],
        "imdb_id": d.get("imdb_id"),
        "title": d["title"],
        "release_date": d.get("release_date"),
        "budget": d.get("budget"),
        "revenue": d.get("revenue"),
        "runtime": d.get("runtime"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])),
        "original_language": d.get("original_language"),
        "popularity": d.get("popularity"),
        "production_companies": "|".join(c["name"] for c in d.get("production_companies", [])),
        "director": "|".join(directors),
        "top_cast": "|".join(c["name"] for c in cast[:5]),
        "vote_count": d.get("vote_count"),
        "vote_average": d.get("vote_average"),   
        "collection_id": collection.get("id"),          # franchise/series ID, blank if none
        "collection_name": collection.get("name"),
        "us_certification": us_certification(d),        # MPAA rating, blank if TMDB has none
    }

OUTFILE = "data/tmdb_movies.csv"

# Resume from a previous run if the file exists
if os.path.exists(OUTFILE):
    existing = pd.read_csv(OUTFILE)
    # An old CSV without the new columns would mix old and new rows, so stop instead
    if "us_certification" not in existing.columns:
        raise RuntimeError(f"{OUTFILE} is from the old version. Rename or move it, then re-run this cell.")
    rows = existing.to_dict("records")
    done = set(existing["tmdb_id"])
    print(f"Resuming with {len(done)} movies already saved")
else:
    rows, done = [], set()

for i, mid in enumerate(movie_ids, 1):
    if mid in done:
        continue
    d = get(f"/movie/{mid}", append_to_response="credits,release_dates", language="en-US")
    if d:
        rows.append(parse_movie(d))
    done.add(mid)

    if i % 500 == 0: 
        pd.DataFrame(rows).to_csv(OUTFILE, index=False)
        print(i, "of", len(movie_ids), "processed")
    time.sleep(0.03)

# Final save so the last partial batch 
pd.DataFrame(rows).to_csv(OUTFILE, index=False)
print(len(rows), "movies saved to", OUTFILE)

500 of 14509 processed
1000 of 14509 processed
1500 of 14509 processed
2000 of 14509 processed
2500 of 14509 processed
3000 of 14509 processed
3500 of 14509 processed
4000 of 14509 processed
4500 of 14509 processed
5000 of 14509 processed
5500 of 14509 processed
6000 of 14509 processed
6500 of 14509 processed
7000 of 14509 processed
7500 of 14509 processed
8000 of 14509 processed
8500 of 14509 processed
9000 of 14509 processed
9500 of 14509 processed
10000 of 14509 processed
10500 of 14509 processed
11000 of 14509 processed
11500 of 14509 processed
12000 of 14509 processed
12500 of 14509 processed
13000 of 14509 processed
13500 of 14509 processed
14000 of 14509 processed
14500 of 14509 processed
14509 movies saved to data/tmdb_movies.csv
